In [1]:
import pandas as pd
from datetime import datetime
import os

In [11]:
# Load the CSV file into a pandas DataFrame
cPath = os.path.dirname(os.path.realpath('./CAE/New/'))
file_path = cPath + 'Hourly5837.csv'
myData = pd.read_csv('Hourly5837.csv', parse_dates=['Start'], date_format='%Y/%m/%d %H:%M')
myData.set_index('Start', drop = False, inplace=True)
print(myData.head)
print(myData.index.name)
myData['Delta'] = 0
# Adding a 'label' column to differentiate between OffPeak, OnPeak, and OnPeakHigh 
myData['label'] = 'OffPeak'
myData.head
print(myData['Start'][0], myData['Start'][1])

<bound method NDFrame.head of                            Start  Usage
Start                                  
10/1/2023 1:00    10/1/2023 1:00   1039
10/1/2023 2:00    10/1/2023 2:00   1016
10/1/2023 3:00    10/1/2023 3:00    953
10/1/2023 4:00    10/1/2023 4:00    804
10/1/2023 5:00    10/1/2023 5:00    856
...                          ...    ...
9/30/2024 20:00  9/30/2024 20:00    866
9/30/2024 21:00  9/30/2024 21:00    788
9/30/2024 22:00  9/30/2024 22:00   1240
9/30/2024 23:00  9/30/2024 23:00   1211
10/1/2024 0:00    10/1/2024 0:00   1126

[8784 rows x 2 columns]>
Start
10/1/2023 1:00 10/1/2023 2:00


/tmp/ipykernel_14017/3642685610.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(myData['Start'][0], myData['Start'][1])


In [10]:
# Find irregular data
def missing_data(data):
    e_td = pd.to_timedelta('1 hour')
    for i in range(0, len(data)-1, 1):
        c_td = data['Start'][i+1] - data['Start'][i]
        if(e_td != c_td): 
           print (data['Start'][i], data['Start'][i+1], c_td)

missing_data(myData)

/tmp/ipykernel_14017/2693035036.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  c_td = data['Start'][i+1] - data['Start'][i]


TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [55]:
# Calculate the derivative of the usage
def calc_delta(data):
    e_td = pd.to_timedelta('15 minutes')
    delta = []
    for i in range(0, len(data)-1, 1):
        dUsage = data['Usage'][i+1] - data['Usage'][i]
        dt = data['Start'][i+1] - data['Start'][i]
        delta.append(dUsage / dt.total_seconds())
    delta.append(0)
    return delta 

myData['Delta'] = calc_delta(myData)

/tmp/ipykernel_5956/609972020.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dUsage = data['Usage'][i+1] - data['Usage'][i]


KeyError: 'Start'

In [63]:
# This needs closer inspection, but it seems to be a way to set the labels based on the month and hour

def set_labels(data):
    data['label'] = 'w'  # Default label as winter
    month_mask = data.index.month.isin([5, 6, 7, 8, 9])  # May to September
    data.loc[month_mask, 'label'] = 's' # Summer months

    # Identify OnPeak based on winter or summer
    time_mask = data.index.hour.isin([18, 19, 20, 21])  # Summer OnPeak hours. Does not include 22
    label_mask = data['label'] == 's'
    data.loc[time_mask & label_mask, 'label'] = 'sOnPeak'  # Summer OnPeak
    time_mask = data.index.hour.isin([6, 7, 8, 9, 18, 19, 20, 21])  # OnPeak hours. Does not include 22
    label_mask = data['label'] == 'w'
    data.loc[time_mask & label_mask, 'label'] = 'wOnPeak'  # Winter OnPeak

    # Identify the OffPeak times
    label_mask = data['label'] == 's'
    data.loc[label_mask, 'label'] = 'OffPeak'  # Summer OffPeak 
    label_mask = data['label'] == 'w'
    data.loc[label_mask, 'label'] = 'OffPeak'  # Winter OffPeak
    label_mask = data.index.day_of_week.isin([5, 6])  # Saturday and Sunday
    data.loc[label_mask, 'label'] = 'OffPeak'  # Weekend OffPeak

    #Identify the high usage periods during OnPeak times
    data.loc[(data['label'] == 'wOnPeak') & (data['Usage'] > 100), 'label'] = 'wOnPeakHigh'
    data.loc[(data['label'] == 'sOnPeak') & (data['Usage'] > 100), 'label'] = 'sOnPeakHigh'
    return data

myData = set_labels(myData)
print('myData: ', myData.value_counts('label'), '\n')

AttributeError: 'Index' object has no attribute 'month'

In [7]:
Y3.to_csv('Y3.labels.csv')
Y4.to_csv('Y4.labels.csv')
Y5.to_csv('Y5.labels.csv')